# two-optimizers-alternating-step — worked example 2: Verify only the stepped network moves

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `two-optimizers-alternating-step`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Because each optimizer is constructed with only one network's parameters, the D-step leaves the generator's weights untouched, and vice versa. Snapshotting weights before and after each phase makes this isolation visible — the core reason two optimizers are used instead of one.

## Worked solution

We prove optimizer isolation by snapshotting weights around each phase.

1. We clone `G`'s and `D`'s first weight before the D-step.
2. We run the D-step (detached fake, D loss, `D_opt.step()`). Only `D`'s weights should change; `G`'s clone should still match.
3. We run the G-step and check the reverse: `G` moved, and the comparison uses fresh snapshots.

The printed booleans confirm that after the D-step the generator was unchanged, and after the G-step the generator did change — exactly the isolation two optimizers provide.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(1)
G = nn.Linear(3, 5)
D = nn.Linear(5, 1)
G_opt = t.optim.SGD(G.parameters(), lr=0.1)
D_opt = t.optim.SGD(D.parameters(), lr=0.1)
z = t.randn(6, 3)
x_real = t.randn(6, 5)

G_before = G.weight.detach().clone()
# D-step
D_opt.zero_grad()
loss_D = (D(G(z).detach()) - D(x_real)).mean()
loss_D.backward(); D_opt.step()
G_unchanged = t.equal(G_before, G.weight.detach())

G_before2 = G.weight.detach().clone()
# G-step
G_opt.zero_grad()
loss_G = -D(G(z)).mean()
loss_G.backward(); G_opt.step()
G_changed = not t.equal(G_before2, G.weight.detach())

print('G unchanged by D-step:', G_unchanged)
print('G changed by G-step:', G_changed)